# Orchestrator Agent

Coordinates the planner and travel agents via a workflow graph.
Uses MCP to discover agents and routes tasks through them in sequence.

In [ ]:
import asyncio
import json
import logging
from collections.abc import AsyncIterable

import uvicorn
from a2a.types import (
    StreamResponse,
    TaskArtifactUpdateEvent,
    TaskState,
    TaskStatusUpdateEvent,
)
from dotenv import load_dotenv
from google import genai
from google.protobuf import json_format

import prompts
from common import BaseAgent, build_a2a_app, init_api_key
from workflow import Status, WorkflowGraph, WorkflowNode

load_dotenv()
init_api_key()
logger = logging.getLogger(__name__)

In [ ]:
class OrchestratorAgent(BaseAgent):
    def __init__(self):
        super().__init__(agent_name='Orchestrator Agent', description='Facilitate inter agent communication', content_types=['text', 'text/plain'])
        self.graph = None
        self.results = []
        self.travel_context = {}
        self.query_history = []
        self.context_id = None

    async def _generate_summary(self) -> str:
        client = genai.Client()
        response = client.models.generate_content(
            model='gemini-2.0-flash',
            contents=prompts.SUMMARY_COT_INSTRUCTIONS.replace('{travel_data}', str(self.results)),
            config={'temperature': 0.0},
        )
        return response.text

    def _answer_user_question(self, question) -> str:
        try:
            client = genai.Client()
            response = client.models.generate_content(
                model='gemini-2.0-flash',
                contents=prompts.QA_COT_PROMPT
                    .replace('{TRIP_CONTEXT}', str(self.travel_context))
                    .replace('{CONVERSATION_HISTORY}', str(self.query_history))
                    .replace('{TRIP_QUESTION}', question),
                config={'temperature': 0.0, 'response_mime_type': 'application/json'},
            )
            return response.text
        except Exception:
            return '{"can_answer": "no", "answer": "Cannot answer based on provided context"}'

    def _set_node_attrs(self, node_id, task_id=None, context_id=None, query=None):
        attrs = {}
        if task_id: attrs['task_id'] = task_id
        if context_id: attrs['context_id'] = context_id
        if query: attrs['query'] = query
        self.graph.set_node_attributes(node_id, attrs)

    def _add_graph_node(self, task_id, context_id, query, node_id=None, node_key=None, node_label=None):
        node = WorkflowNode(task=query, node_key=node_key, node_label=node_label)
        self.graph.add_node(node)
        if node_id:
            self.graph.add_edge(node_id, node.id)
        self._set_node_attrs(node.id, task_id, context_id, query)
        return node

    def _clear_state(self):
        self.graph = None
        self.results.clear()
        self.travel_context.clear()
        self.query_history.clear()

    async def stream(self, query, context_id, task_id) -> AsyncIterable[dict[str, any]]:
        if not query:
            raise ValueError('Query cannot be empty')
        if self.context_id != context_id:
            self._clear_state()
            self.context_id = context_id

        self.query_history.append(query)
        start_node_id = None

        if not self.graph:
            self.graph = WorkflowGraph()
            planner_node = self._add_graph_node(task_id, context_id, query, node_key='planner', node_label='Planner')
            start_node_id = planner_node.id
        elif self.graph.state == Status.PAUSED:
            start_node_id = self.graph.paused_node_id
            self._set_node_attrs(start_node_id, query=query)

        while True:
            self._set_node_attrs(start_node_id, task_id=task_id, context_id=context_id)
            should_resume = False

            async for stream_resp, task in self.graph.run_workflow(start_node_id):
                payload_type = stream_resp.WhichOneof('payload')

                if payload_type == 'status_update':
                    evt = stream_resp.status_update
                    context_id = evt.context_id
                    if evt.status.state == TaskState.TASK_STATE_COMPLETED and context_id:
                        continue
                    if evt.status.state == TaskState.TASK_STATE_INPUT_REQUIRED:
                        question = evt.status.message.parts[0].text
                        try:
                            answer = json.loads(self._answer_user_question(question))
                            if answer['can_answer'] == 'yes':
                                query = answer['answer']
                                start_node_id = self.graph.paused_node_id
                                self._set_node_attrs(start_node_id, query=query)
                                should_resume = True
                        except Exception:
                            pass

                elif payload_type == 'artifact_update':
                    artifact = stream_resp.artifact_update.artifact
                    self.results.append(artifact)
                    if artifact.name == 'PlannerAgent-result':
                        artifact_data = json_format.MessageToDict(artifact.parts[0].data)
                        if 'trip_info' in artifact_data:
                            self.travel_context = artifact_data['trip_info']
                        current_node_id = start_node_id
                        for idx, task_data in enumerate(artifact_data['tasks']):
                            node = self._add_graph_node(task_id, context_id, task_data['description'], node_id=current_node_id)
                            current_node_id = node.id
                            if idx == 0:
                                should_resume = True
                                start_node_id = node.id
                    else:
                        continue

                if not should_resume:
                    yield (stream_resp, task)

            if not should_resume:
                break

        if self.graph.state == Status.COMPLETED:
            summary = await self._generate_summary()
            self._clear_state()
            yield {'response_type': 'text', 'is_task_complete': True, 'require_user_input': False, 'content': summary}

In [ ]:
app = build_a2a_app(OrchestratorAgent(), 'agent_cards/orchestrator_agent.json')

config = uvicorn.Config(app=app, host='localhost', port=10101, log_level='info')
server = uvicorn.Server(config)
asyncio.ensure_future(server.serve())
print('Orchestrator Agent running at http://localhost:10101')